# Exercice 5 : Reconnaissance active avec nmap

> **En une phrase :** on envoie de **vrais paquets** à une cible pour savoir quels services elle expose, et en quelle version. C'est précis, mais **visible**.

**⚠️ Règle du cours.** La seule cible autorisée est **scanme.nmap.org** : le projet Nmap la met à disposition pour s'exercer (scans légers, quelques fois par jour). Scanner une autre machine sans autorisation écrite est illégal (Code criminel, art. 342.1).

**Le plan.**
1. Installer nmap.
2. Ce que Shodan disait de la cible (passif).
3. Scanner les ports (actif).
4. Lire les versions, et ce qu'elles valent.
5. Lire une bannière à la main.
6. Ce que la cible a vu de nous.

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
# Installe nmap sur Colab (déjà présent en local, sinon : apt install nmap)
!which nmap > /dev/null || (apt-get -qq update > /dev/null && apt-get -qq install -y nmap > /dev/null)
!nmap --version | head -1
import requests, socket, subprocess, os
cible = "scanme.nmap.org"    # NE PAS MODIFIER : seule cible autorisée
print('Prêt')

Nmap version 7.94SVN ( https://nmap.org )


Prêt


## Étape 1 : Ce que Shodan disait de la cible (passif)
**Ce qu'on fait :** avant d'agir, on regarde ce qu'on sait déjà sans se montrer (ex. 4).

In [2]:
ip = socket.gethostbyname(cible)
passif = requests.get(f"https://internetdb.shodan.io/{ip}", timeout=10).json()
print("IP           :", ip)
print("Ports Shodan :", passif["ports"])
print("Logiciels    :", passif["cpes"])

IP           : 45.33.32.156
Ports Shodan : [22, 80, 123]
Logiciels    : ['cpe:/a:openbsd:openssh:6.6.1p1', 'cpe:/a:apache:http_server:2.4.7', 'cpe:/a:ntp:ntp:3', 'cpe:/o:canonical:ubuntu_linux']


**👀 Ce qu'on doit voir :** IP **45.33.32.156**, ports **22, 80, 123**, et des logiciels : OpenSSH 6.6.1p1, Apache 2.4.7, Ubuntu.

## Étape 2 : Scanner les ports (actif)
**Ce qu'on fait :** `nmap -F` teste les **100 ports** les plus courants. `-Pn` : ne pas vérifier d'abord si la machine répond au ping.

In [3]:
!nmap -Pn -F {cible}

Starting Nmap 7.94SVN ( https://nmap.org ) at 2026-11-03 17:57 EST


Nmap scan report for scanme.nmap.org (45.33.32.156)
Host is up (0.071s latency).
Not shown: 98 closed tcp ports (conn-refused)
PORT   STATE SERVICE
22/tcp open  ssh
80/tcp open  http

Nmap done: 1 IP address (1 host up) scanned in 0.13 seconds


**👀 Ce qu'on doit voir :** **22/tcp open ssh** et **80/tcp open http**. Les 98 autres ports sont fermés. Moins d'une seconde.

**❓ Question pour la classe :** Shodan voyait le port **123**, nmap ne le voit pas. Qui a raison ?

*Réponse : les deux. Le port 123 est **NTP** (synchronisation de l'heure), qui fonctionne en **UDP**. Par défaut, nmap ne teste que **TCP**. La cellule suivante teste l'UDP (il faut être administrateur : ça marche sur Colab).*

In [4]:
if os.geteuid() == 0:                    # sur Colab, on est administrateur (root)
    !nmap -Pn -sU -p 123 {cible}
else:
    print("Scan UDP : nécessite les droits administrateur. Exécutez cette cellule sur Colab.")

Scan UDP : nécessite les droits administrateur. Exécutez cette cellule sur Colab.


**👀 Ce qu'on doit voir (sur Colab) :** **123/udp open ntp**. Shodan avait raison : toujours vérifier le protocole.

## Étape 3 : Lire les versions
**Ce qu'on fait :** `-sV` demande à chaque service de se présenter, et en déduit le logiciel et sa version.

In [5]:
!nmap -Pn -sV -p 22,80 {cible}

Starting Nmap 7.94SVN ( https://nmap.org ) at 2026-11-03 17:57 EST


Nmap scan report for scanme.nmap.org (45.33.32.156)
Host is up (0.075s latency).

PORT   STATE SERVICE VERSION
22/tcp open  ssh     OpenSSH 6.6.1p1 Ubuntu 2ubuntu2.13 (Ubuntu Linux; protocol 2.0)
80/tcp open  http    Apache httpd 2.4.7 ((Ubuntu))
Service Info: OS: Linux; CPE: cpe:/o:linux:linux_kernel

Service detection performed. Please report any incorrect results at https://nmap.org/submit/ .
Nmap done: 1 IP address (1 host up) scanned in 7.27 seconds


**👀 Ce qu'on doit voir :** **OpenSSH 6.6.1p1 Ubuntu** et **Apache httpd 2.4.7**. Deux versions de **2014**, plus de 10 ans sans mise à jour (c'est voulu : c'est une machine d'exercice).

**❓ Question pour la classe :** à l'ex. 4, Shodan listait des CVE pour cette machine. Maintenant qu'on a la version **vérifiée** d'Apache, que fait un attaquant ?

*Réponse : il cherche les CVE qui touchent **précisément** Apache 2.4.7 (par exemple sur cvedetails.com ou la base NVD), puis un exploit public. La version exacte transforme une liste de 140 CVE « possibles » en quelques vulnérabilités réelles.*

## Étape 4 : Lire une bannière à la main
**Ce qu'on fait :** pas besoin de nmap pour cela : on ouvre une connexion sur le port 22, et le serveur SSH **se présente tout seul**.

In [6]:
import time
banniere = ""
for essai in range(3):                   # la cible ignore parfois les connexions trop rapprochées
    with socket.create_connection((cible, 22), timeout=10) as connexion:
        banniere = connexion.recv(100).decode().strip()
    if banniere:
        break
    time.sleep(3)
print("Le serveur dit :", banniere or "(rien : réessayez dans quelques secondes)")

Le serveur dit : SSH-2.0-OpenSSH_6.6.1p1 Ubuntu-2ubuntu2.13


**👀 Ce qu'on doit voir :** **SSH-2.0-OpenSSH_6.6.1p1 Ubuntu-2ubuntu2.13**. La version complète, envoyée à quiconque se connecte.

## Étape 5 : Ce que la cible a vu de nous
**Ce qu'on fait :** on affiche **notre** adresse IP publique. C'est elle qui apparaît maintenant dans les journaux de scanme.nmap.org, avec l'heure de chaque paquet.

In [7]:
mon_ip = requests.get("https://api.ipify.org", timeout=10).text
print("Votre IP publique, maintenant dans les journaux de la cible :", mon_ip)

Votre IP publique, maintenant dans les journaux de la cible : 132.212.102.190


**👀 Ce qu'on doit voir :** l'adresse de Google (sur Colab) ou celle de votre réseau. En passif (ex. 1 à 4), la cible n'a jamais vu cette adresse.

## Contre-mesures (côté défenseur)
- **Fermer** ce qui n'a pas besoin d'être ouvert : chaque port ouvert est une porte.
- **Mettre à jour** : la version annoncée sert directement à choisir l'exploit.
- **Masquer les versions** dans les bannières (ex. `ServerTokens Prod` pour Apache). Ça ne corrige rien, mais ça ralentit l'attaquant.
- **Détecter les scans** : 100 ports testés en 2 secondes depuis la même IP, c'est une signature claire. C'est le sujet de la **séance 9**.

## Ce qu'il faut retenir
- La reconnaissance **active** donne une information **fraîche et vérifiée** (versions réelles, protocoles UDP).
- Elle **laisse des traces** : notre IP est maintenant dans les journaux de la cible.
- Elle n'est légale qu'avec une **autorisation** et un **périmètre** écrits.

**Exercice suivant :** on assemble tout ce qu'on a trouvé en un rapport utile au défenseur.